# Geometric programming for inductor design
This notebook aims to replicate the paper titled _Optimization of Inductor Circuits via Geometric Programming_ by Hershenon et al using python.

## Shunt resistance $R_p$
We need to find an expression for $k_6$ so we can express $R_p$ as a monomial.

Starting from the formula for $R_p$
$$
R_p = \frac{1 + [\omega R_\mathrm{si} (C_\mathrm{si} + C_\mathrm{ox}) ]^2}{\omega^2R_\mathrm{si}C_\mathrm{ox}^2}
$$
we substitute the capacitances and resistances for their corresponding monomial expressions
$$
C_\mathrm{ox} = k_2lw
$$
$$
C_\mathrm{si} = k_4lw
$$
$$
R_\mathrm{si} = \frac{k_5}{lw}
$$
For simplicity, let $x=lw$.


In [11]:
import sympy as sp

# Symbols
k2, k4, k5, omeg = sp.symbols('k2 k4 k5 omega', positive=True)
x = sp.symbols('x', positive=True)  # x = l*w

# Monomial models
Cox = k2 * x
Csi = k4 * x
Rsi = k5 / x

# Original Rp expression from the paper
Rp = (1 + (omeg*Rsi*(Csi+Cox))**2)/(omeg**2*Rsi*Cox**2)
# Simplify expression
Rp_simplified = sp.simplify(Rp)

print("Rp simplified:")
sp.pprint(Rp_simplified)

Rp simplified:
  2  2          2    
k₅ ⋅ω ⋅(k₂ + k₄)  + 1
─────────────────────
       2     2       
     k₂ ⋅k₅⋅ω ⋅x     


Now we have
$$
R_p = \frac{k_5^2\omega^2(k_2+k_4)^2+1}{k_2^2k_5\omega^2x}
$$

In [12]:
k6 = sp.simplify(Rp_simplified.coeff(x, -1))
print("k6:")
k6 = sp.simplify(k6)

print(sp.latex(k6))

k6:
\frac{k_{5} \left(k_{2} + k_{4}\right)^{2}}{k_{2}^{2}} + \frac{1}{k_{2}^{2} k_{5} \omega^{2}}


This way we obtain
$$
k_6=
\frac{k_{5} \left(k_{2} + k_{4}\right)^{2}}{k_{2}^{2}} + \frac{1}{k_{2}^{2} k_{5} \omega^{2}}
$$

## Shunt capacitance $C_p$
We need to find expressions for $k_7$ and $k_8$ so we can express $C_p$ as a posynomial.

Starting from the formula for $C_p$
$$
C_p = \frac{C_\mathrm{ox}+\omega^2R_\mathrm{si}^2(C_\mathrm{si} + C_\mathrm{ox})C_\mathrm{si}C_\mathrm{ox}}{1 + [\omega R_\mathrm{si}(C_\mathrm{si} + C_\mathrm{ox})]^2}
$$
_Note: The original formula from the paper contains an error: the $R_\mathrm{si}$ term in the numerator should be squared (as it is here)._

First, substitute the capacitances and resistances by their corresponding monomial expressions
$$
C_\mathrm{ox} = k_2lw
$$
$$
C_\mathrm{si} = k_4lw
$$
$$
R_\mathrm{si} = \frac{k_5}{lw}
$$
For simplicity, let $x=lw$.

In [13]:
# Original Cp expression from the paper
Cp = (Cox + omeg**2 * Rsi**2 * (Csi + Cox) * Csi * Cox) / (1 + (omeg * Rsi * (Csi + Cox))**2)

# Simplify expression
Cp_simplified = sp.simplify(Cp)

print("Cp simplified:")
sp.pprint(Cp_simplified)

Cp simplified:
     ⎛     2  2              ⎞
k₂⋅x⋅⎝k₄⋅k₅ ⋅ω ⋅(k₂ + k₄) + 1⎠
──────────────────────────────
      2  2          2         
    k₅ ⋅ω ⋅(k₂ + k₄)  + 1     


Now we have
$$
C_p = \frac{k_2 x \left(k_4k_5^2\omega^2(k_2+k_4) + 1\right)}{k_5^2\omega^2(k_2 + k_4)^2 + 1}
$$
and we can treat the expression as a polynomial in $x$ to get the coefficients.

In [14]:
Cp_series = sp.series(Cp, x, 0, 3).removeO()

k7 = sp.simplify(Cp_series.coeff(x, 1))
k8 = sp.simplify(Cp_series.coeff(x, 2))

print("k7:")
k7 = sp.simplify(k7)

print(sp.latex(k7))
#print(sp.latex(sp.simplify(k7)))
print("k8:")
sp.pprint(k8)

k7:
\frac{k_{2} \left(k_{2} k_{4} k_{5}^{2} \omega^{2} + k_{4}^{2} k_{5}^{2} \omega^{2} + 1\right)}{k_{5}^{2} \omega^{2} \left(k_{2} + k_{4}\right)^{2} + 1}
k8:
0


This way we obtain
$$ 
k_7= \frac{k_{2} \left(k_{2} k_{4} k_{5}^{2} \omega^{2} + k_{4}^{2} k_{5}^{2} \omega^{2} + 1\right)}{k_{2}^{2} k_{5}^{2} \omega^{2} + 2 k_{2} k_{4} k_{5}^{2} \omega^{2} + k_{4}^{2} k_{5}^{2} \omega^{2} + 1}
$$

## Q factor
The $Q$ factor is defined as
$$
Q = \frac{\mathrm{Im}(Z)}{\mathrm{Re}(Z)},
$$
where $Z$ is the is the driving point impedance between the terminals. To get this impedance we can express the equivalent circuit as:

<p align="center">
<img src="imag/TwoPort.png">
</p>

The current entering port 1 is
$$
I_1 = V_1 Y_p + Y_s(V_1 - V_2) + \frac{V_1-V_2}{Z_s}
$$
and the current that exits port 2 is
$$
I_2 = V_2Y_p + Y_s(V_2 - V_1) + \frac{V_2-V_1}{Z_s},
$$
thus
$$
V = V_1 - V_2.
$$
And beacuse this is a reciprocal network
$$
V_1 = \frac{V}{2},\qquad V_2 = -\frac{V}{2}
$$
$$
I = I_1 = -I_2.
$$

Now we can write $I_1$ as
$$
I = Y_p\frac{V}{2} + Y_sV + \frac{V}{Z_s},
$$
and find the impedance between terminals
$$
Z = \frac{V}{I} = \frac{1}{\frac{Y_p}{2} + Y_s + \frac{1}{Z_s}}
$$


In [15]:
Rs, Ls, Cs, Rp, Cp = sp.symbols("R_s, L_s, C_s, R_p, C_p", positive=True)
V, I = sp.symbols("V, I")

Zs = Rs + sp.I*omeg*Ls
Yp = 1/Rp + sp.I*omeg*Cp
Ys = sp.I*omeg*Cs

I = V/2*Yp + Ys*V+V/Zs
Zdiff = V/I

Q = sp.im(Zdiff)/sp.re(Zdiff)
Q = sp.simplify(sp.expand(Q))
sp.print_latex(Q)

\frac{R_{p} \omega \left(- C_{p} L_{s}^{2} \omega^{2} - C_{p} R_{s}^{2} - 2 C_{s} L_{s}^{2} \omega^{2} - 2 C_{s} R_{s}^{2} + 2 L_{s}\right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}}


From the expression we obtained
$$
Q = \frac{R_{p} \omega \left(- C_{p} L_{s}^{2} \omega^{2} - C_{p} R_{s}^{2} - 2 C_{s} L_{s}^{2} \omega^{2} - 2 C_{s} R_{s}^{2} + 2 L_{s}\right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}}
$$
we can factor the numerator
$$
Q = \frac{R_{p} \omega \left(2 L_{s} - (C_{p} + 2C_s) (L_{s}^{2} \omega^{2} + R_s^2) \right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}},
$$
and reorder
$$
Q = \frac{\omega L_{s}}{R_s}\cdot\frac{ 2R_{p}  \left(1  - \frac{R_s^2\frac{C_{p} + 2C_s}{2}}{L_s} - \omega^{2}L_{s}\frac{C_{p} + 2C_s}{2} \right)}{2R_{p}+\left[\left(\frac{L_{s} \omega}{R_s}\right)^{2} +1\right]R_{s}},
$$
to see that in order to match the $Q$ factor expression in the paper, we must define
$$
C_\mathrm{tot} = C_p + 2C_s
$$

## Geometric programming

In [16]:
# Turn sympy expressions into functions
k6_func = sp.lambdify((k2, k4, k5, omeg), k6, modules="math")
k7_func = sp.lambdify((k2, k4, k5, omeg), k7, modules="math")
k8_func = sp.lambdify((k2, k4, k5, omeg), k8, modules="math")

In [17]:

from gpkit import Variable, Model, ureg
import numpy as np

# Parameters
mu_0 = Variable("\\mu_0", 4*np.pi*1e-7, "H/m", "Magnetic permeability of free space", constant=True)
omega = Variable("\\omega", 2*np.pi*2.5e9, "rad/s", "Operating angular frequency", constant=True)
e_0= Variable("e_{0}", 8.854187e-12, "F/m", "Electric permittivity of free space", constant=True)

beta = 1.33e-3#Variable("beta", 1.33e-3, "-", "Beta coefficient for octagonal geommetry", constant=True)
alpha_1 = -1.21 #Variable("alpha_1", -1.21,  "-", "Alpha coefficient for outter diameter in octagonal geommetry", constant=True)
alpha_2 = -0.163 #Variable("alpha_2", -0.163, "-", "Alpha coefficient for track width in octagonal geommetry", constant=True)
alpha_3 = 2.43 #Variable("alpha_3",  2.43,  "-", "Alpha coefficient for average diameter in octagonal geommetry", constant=True)
alpha_4 = 1.75 #Variable("alpha_4",  1.75,  "-", "Alpha coefficient for number of turns in octagonal geommetry", constant=True)
alpha_5 = -0.049 #Variable("alpha_5", -0.049, "-", "Alpha coefficient for track spacing in octagonal geommetry", constant=True)

L_req = Variable("L_{req}", 5, "nH", "Required inductance in nano Henries", constant=True)
Q_min = Variable("Q_{min}", "-", "Minimum quality factor", positive=True)
omega_sr_min = Variable("\\omega_{sr,min}", 2*np.pi*7e9, "rad/s", "Minimum self-resonance frequency", constant=True)

sigma_tm2 = Variable("\\sigma_{tm2}", 30.3e6, "S/m", "Conductivity of TopMetal2 for ihp SG13G2", constant=True)
t_tm2 = Variable("t_{m2}", 3e-6, "m", "Thickness of TopMetal2 for ihp SG13Gw", constant=True)

e_ox = Variable("e_{ox}", 4.1*8.854187e-12, "F/m", constant=True)
t_ox = Variable("t_{ox}", 2.8e-6 + 0.85e-6 + 0.54e-6 + 0.54e-6 + 0.54e-6 + 0.54e-6 + 0.64e-6 , "m", "Oxide thickness between substrate and TopMetal2", constant=True)
t_ox_tm1tm2 = Variable("t_{ox,TM1-TM2}", 2.8e-6, "m", "Oxide thickness between TopMetal1 and TopMetal2", constant=True)

t_sub = Variable("t_{sub}", 280e-6, "m", "Substrate thickness", constant=True)
e_sub = Variable("e_{sub}", 11.9*8.854187e-12, "F/m", constant=True)
sigma_sub = Variable("\\sigma_{sub}", 2, "S/m", "Conductivity of substrate for ihp SG13G2", constant=True)

w_min = Variable("w_{min}", 2e-6, "m", "Minimum TopMetal2 width", constant=True)
s_min = Variable("s_{min}", 2e-6, "m", "Minimum TopMetal2 separation", constant=True)
A_max = Variable("A_max", (250e-6)**2, "m^2", "Maximum TopMetal2 area", constant=True)

# Variables
d_out = Variable("d_{out}", "m", "Outter diameter of the inductor", positive=True)
d_in = Variable("d_{in}", "m", "Inner diameter of the inductor", positive=True)
#d_avg = Variable("d_{avg}", "m", "Average diameter of the inductor", positive=True)
#d_avg = (d_out + d_in)/2
d_avg = (d_out * d_in)**0.5
w = Variable("w", "m", "Metal track width", positive=True)
s = Variable("s", "m", "Spacing between metal tracks", positive=True)
n = Variable("n", "-", "Number of turns", positive=True)
#l = Variable("l", "m", "length of the conductor", positive=True)

# Monomials
skin_depth_tm1 = np.sqrt(2/(omega.value * mu_0.value * sigma_tm2.value))
l = 8 * d_avg * n/ (1 + 2**0.5) # Conductor length (8 sides of an octagon with a span of d_avg)

if (skin_depth_tm1 < t_tm2.value):
    print(f"Skin depth is {skin_depth_tm1.to("um")}")
    k_1 = 1 / (sigma_tm2 * skin_depth_tm1*(1 - np.exp(-t_tm2.value/skin_depth_tm1)))
else:
    print("No skin effect")
    k_1 = 1 / (sigma_tm2 * t_tm2)  # No skin effect occurs
k_2 = (e_ox )/(2 * t_ox)
k_3 = (e_ox)/(t_ox_tm1tm2)
k_4 = e_sub / (2 * t_sub)
k_5 = 2 * t_sub / sigma_sub

# Design variables
R_s  = k_1 * l /w 
C_ox = k_2 * l * w
C_s  = k_3 * n * w**2 # NOTE: This is not valid for pcLab's geometry
C_si = k_4 * l * w
R_si = k_5 / (l * w)


# L must be adjusted since the paper uses dimensionless units
um = Variable("um", 1e-6, "m", constant=True)

L = beta * (d_out/um)**alpha_1 \
        * (w/um)**alpha_2 \
        * (d_avg/um)**alpha_3 \
        * n**alpha_4 \
        * (s/um)**alpha_5 \
        * Variable("nH", 1e-9, "H", constant=True)

# Equivalent model
F_per_m2 = Variable("F_per_m2", 1, "F/m^2", constant=True)
ohm_m2   = Variable("ohm_m2", 1, "ohm*m^2", constant=True)

k_6 = k6_func(k_2.to("F/m^2").value.magnitude, k_4.to("F/m^2").value.magnitude, k_5.to("ohm*m^2").value.magnitude, omega.value.magnitude) * ohm_m2
k_7 = k7_func(k_2.to("F/m^2").value.magnitude, k_4.to("F/m^2").value.magnitude, k_5.to("ohm*m^2").value.magnitude, omega.value.magnitude) * F_per_m2
k_8 = k8_func(k_2.to("F/m^2").value.magnitude, k_4.to("F/m^2").value.magnitude, k_5.to("ohm*m^2").value.magnitude, omega.value.magnitude)# * Variable("k8_units", 1, "F/m^4")

# Fix units


R_p = k_6/(l * w) 
C_p = k_7 * l * w 
C_tot = C_p + 2*C_s

Skin depth is 1.8286425166069231 micrometer


In [18]:
objective = Q_min**-1

# Normalized variables
rho = omega * L / R_s  # inductive reactance
gamma = omega**2 * L * C_tot  # Capacitive loading
delta = R_s**2 * C_tot / L  # Q factor normalization

k_sr = omega_sr_min / omega

constraints = [
    L == L_req,  # Required inductance

     # normalized Q constraint
    Q_min * (2*R_p + (rho**2 + 1)*R_s) / (rho * 2*R_p) + delta + gamma <= 1,

    # self resonance
    delta/2 + gamma/2 <= 1,

    # minimum SRF
    k_sr**2 * gamma/2 + delta/2 <= 1,
    
    # PDK dimensional constraints
    w >= w_min,
    s >= s_min,
    d_out**2 <= A_max,

    #Geometry constriants
    d_out >= d_in + 2*n*(w+s),
    d_in >= w
]

m = Model(objective, constraints)

#m.debug()
sol = m.solve()
print(sol.table())

Using solver 'cvxopt'
 for 6 free variables
  in 11 posynomial inequalities.
Solving took 0.0137 seconds.

          ┃┓
     Cost╺┫┃
 (0.0801) ┃┣╸Q_{min}^-1
          ┃┛ (0.0801)



       ┃┓
       ┃┃
       ┃┣╸d_{out} ≥ d_{in} + 2·n·(w + s)
       ┃┛
       ┃┓
       ┃┣╸um = 1e-06m
       ┃┛
       ┃┣╸Q_{min}·2·0.000321·ohm_m2/(8·(d_{out}·d_{in})^0.5·n/2.41·w) + ((\o…
       ┃┛
 Model╺┫┓
       ┃┣╸nH = 1e-09H
       ┃┛
       ┃┣╸\sigma_{tm2} = 3.03e+07S/m
       ┃┛
       ┃┣╸\omega = 1.57e+10rad/s
       ┃┛
       ┃┣╸A_max = 6.25e-08m²
       ┃┣╸A_max ≥ d_{out}²
       ┃┣╸[8 terms]
       ┃┛


Free Variables
--------------
Q_{min} : 12.49          Minimum quality factor
 d_{in} : 0.000122   [m] Inner diameter of the inductor
d_{out} : 0.00025    [m] Outter diameter of the inductor
      n : 4.93           Number of turns
      s : 2e-06      [m] Spacing between metal tracks
      w : 1.098e-05  [m] Metal track width

Fixed Variables
---------------
          A_max : 6.25e-08   [m²]  

In [19]:
print(f"""
L={L.sub(sol["variables"]).value.to("nH")}
w={w.sub(sol["variables"]).value.to("um")}
s={s.sub(sol["variables"]).value.to("um")}
l={l.sub(sol["variables"]).value.to("um")}
n={n.sub(sol["variables"]).value} turns
d_out={d_out.sub(sol["variables"]).value.to("um")}
d_in={d_in.sub(sol["variables"]).value.to("um")}
d_avg={d_avg.sub(sol["variables"]).value.to("um")}
C_si={C_si.sub(sol["variables"]).value.to("fF")}
C_ox={C_ox.sub(sol["variables"]).value.to("fF")}
C_s={C_s.sub(sol["variables"]).value.to("fF")}
C_p={C_p.sub(sol["variables"]).value.to("fF")}
R_si={R_si.sub(sol["variables"]).value.to("kohm")}
R_s={R_s.sub(sol["variables"]).value.to("ohm")}
R_p={R_p.sub(sol["variables"]).value.to("kohm")}
""")


L=5.000000290335559 nanohenry
w=10.979570959199064 micrometer
s=2.000001245205167 micrometer
l=2853.2628984015137 micrometer
n=4.929818806060798 turns
d_out=250.0000220954527 micrometer
d_in=122.02617933089552 micrometer
d_avg=174.6612364806443 micrometer
C_si=5.89433457157375 femtofarad
C_ox=88.15967804268992 femtofarad
C_s=7.705063323275268 femtofarad
C_p=5.996157626649758 femtofarad
R_si=8.93780493969044 kiloohm
R_s=5.81809146607647 ohm
R_p=10.231260957233301 kiloohm



In [20]:
L_val  = L.sub(sol["variables"]).value
Ctot_val = C_tot.sub(sol["variables"]).value
Rs_val = R_s.sub(sol["variables"]).value

omega_sr = np.sqrt(2/(L_val * Ctot_val) - Rs_val**2/L_val**2)
f_sr = omega_sr/(2*np.pi)

print("Self resonance frequency =", f_sr.to("GHz"))

Q_min_val = Q_min.sub(sol["variables"]).value
print("Q factor =", Q_min_val)

Self resonance frequency = 21.755232812568032 gigahertz
Q factor = 12.490095363150154


## GDS setup